# KLA SEMICON 2026 — restoration training (Kaggle)

**Before running:**
1. Attach two Kaggle Datasets as Input, named exactly `kla-semicon-2026` (the official archive, zip or already-extracted -- either works) and `kla-code` (this repo folder). Cell 1 finds them by name, so the exact mount path Kaggle uses doesn't matter.
2. Settings → Accelerator → **GPU T4 x2** or **GPU P100**. We deliberately use a *single* GPU (see cell 2).
3. Settings → Internet **On** (needed for the optional `lpips` install, and for `git clone` if the code dataset isn't attached).

Everything else is driven by `configs/base.yaml` — no hyperparameters are set in this notebook.

In [ ]:
# Locates datasets by NAME rather than hardcoded path: different Kaggle
# environments mount inputs at different paths (classic
# /kaggle/input/<name> vs the newer /kaggle/input/datasets/<owner>/<name>),
# so searching avoids having to know which one applies to this account.
CODE_DATASET_NAME = "kla-code"
DATA_DATASET_NAME = "kla-semicon-2026"
REPO_URL = "https://github.com/23f2001033/kla-restore.git"  # fallback once the repo is public
BUDGET_HOURS = 6.5

import os
import shutil
import subprocess
from pathlib import Path


def find_kaggle_dataset(name: str) -> Path:
    root = Path("/kaggle/input")
    direct = root / name
    if direct.is_dir():
        return direct
    matches = [p for p in root.glob(f"*/{name}") if p.is_dir()]
    matches += [p for p in root.glob(f"datasets/*/{name}") if p.is_dir()]
    if matches:
        return matches[0]
    raise FileNotFoundError(
        f"Kaggle dataset '{name}' not found under {root}. "
        f"Top-level contents: {[p.name for p in root.iterdir()] if root.is_dir() else 'N/A'}"
    )


def find_repo_root(base: Path) -> Path:
    """A dragged folder is often wrapped in an extra directory of the same
    name, so train.py can land one (or more) levels deeper than expected.
    Walk down through single-child directories until train.py is found."""
    for _ in range(4):
        if (base / "train.py").is_file():
            return base
        subs = [d for d in base.iterdir() if d.is_dir()]
        if len(subs) != 1:
            break
        base = subs[0]
    return base


WORK = Path("/kaggle/working/kla-restore")
if WORK.exists():
    shutil.rmtree(WORK)  # always start clean -- never build on a stale/partial copy

try:
    code_ds = find_kaggle_dataset(CODE_DATASET_NAME)
    src_root = find_repo_root(code_ds)
    shutil.copytree(src_root, WORK)
    print("code from dataset:", src_root)
except FileNotFoundError as exc:
    print(f"[code dataset not found ({exc}); falling back to git clone]")
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(WORK)], check=True)
    print("code cloned from", REPO_URL)

os.chdir(WORK)
assert (WORK / "train.py").is_file(), f"train.py not found under {WORK}"
rev = subprocess.run(
    ["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True
).stdout.strip()
print("cwd:", os.getcwd(), "| git:", rev or "n/a (dataset copy)")

# Kaggle auto-extracts an uploaded .zip by default, so the dataset may be the
# zip file itself OR an already-extracted train/GT + train/NoisyLR tree.
# src.data.pack_dataset() (used in the next cell) handles either.
DATA_ROOT = find_kaggle_dataset(DATA_DATASET_NAME)
DATA_SOURCE = DATA_ROOT / "train.zip" if (DATA_ROOT / "train.zip").is_file() else DATA_ROOT
print(
    "data source:", DATA_SOURCE,
    "|", "zip file" if DATA_SOURCE.suffix == ".zip" else "extracted directory",
)

In [ ]:
# Pin to a single GPU. DataParallel across T4 x2 gives a modest speedup at the
# cost of a whole class of hangs/desyncs — not a risk worth taking on a run we
# only get to do once, overnight, against a deadline.
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f"{p.name}, {p.total_memory/1e9:.1f} GB")

In [ ]:
!pip -q install lpips  # optional: only used for the final fine-tune + reporting

In [ ]:
# Pack the data into contiguous memmaps (~2 min). Handles both a .zip archive
# and an already-extracted directory (Kaggle auto-extracts uploads by
# default). Re-runnable if the session restarts -- skips work that's done.
import sys; sys.path.insert(0, ".")
from src.data import pack_dataset
meta = pack_dataset(DATA_SOURCE, "/kaggle/working/packed")

In [ ]:
# Pre-flight gate: overfit 2 pairs. If this does not clear 35 dB the pipeline is
# broken and the long run would be wasted. ~1-2 minutes on a GPU.
# Do NOT pipe this through `tail` -- the exit code is the machine-readable result.
!python train.py --overfit 2 --steps 2000 --gate_db 35 --num_workers 2 --no_lpips --data_dir /kaggle/working/packed

In [ ]:
# Main run. train.py measures its own throughput over the first 200 steps and
# sizes the cosine schedule to fit BUDGET_HOURS, so the LR always lands at its
# minimum exactly when the budget runs out.
# Checkpoints + validation every 5k steps -> weights/best.pt survives a crash.
!python train.py --config configs/base.yaml \
    --data_dir /kaggle/working/packed \
    --out_dir /kaggle/working/weights \
    --hours {BUDGET_HOURS} \
    --num_workers 2

In [ ]:
!python evaluate.py --weights /kaggle/working/weights/best.pt \
    --data_dir /kaggle/working/packed --out_dir /kaggle/working/results

In [ ]:
# End-to-end inference timing on the validation split, measured the same way
# KLA measures it: process startup through to the last file written.
import json, numpy as np, os, sys
sys.path.insert(0, ".")
from src.data import split_indices
meta = json.load(open("/kaggle/working/packed/meta.json"))
_, val_idx = split_indices(meta["n"])
lr = np.load("/kaggle/working/packed/lr.npy", mmap_mode="r")
os.makedirs("/kaggle/working/bench_in", exist_ok=True)
for i in val_idx:
    np.save(f"/kaggle/working/bench_in/{i:06d}.npy", np.asarray(lr[i]))
print(len(val_idx), "files staged")

In [ ]:
!time python inference.py --input_dir /kaggle/working/bench_in \
    --output_dir /kaggle/working/bench_out \
    --weights /kaggle/working/weights/best.pt

In [ ]:
# Bundle the checkpoint + logs for download.
!cd /kaggle/working && zip -r submission_artifacts.zip weights results -x '*.pyc'
print("download /kaggle/working/submission_artifacts.zip")